# The Repair: ZymCTRL Re-tested at Matched Relative Push Strength

## Why this notebook exists

`34-ai4dd-residual-norm-audit.ipynb` measured what "1x" (`REFERENCE_NORM = 583.998`, injected as
an absolute vector norm) actually meant in relative terms across the five models this project
tested. The result (`notes/locked-results.md` §1k-FLAG, confirmed 2026-08-23): ZymCTRL's
residual-stream norm at layer 12 is ~51, so an absolute push of 584 there was **α_rel ≈ 11.4** --
11.4x the size of the hidden state it was added to. ProtGPT2's own "1x" at the same layer was
α_rel ≈ 0.17-0.20. Those are not the same experiment at two scales; they are different
interventions that happened to share a label. `26-ai4dd-zymctrl-fair-test.ipynb`'s locked numbers
(§1h) are not wrong as measurements, but the claim "ZymCTRL shows the same pattern as ProtGPT2 at
matched strength" cannot be supported by them, because the strength was never matched.

This notebook is the actual repair: rebuild ZymCTRL's fair-test vectors exactly as `26` did, but
scale them to match ProtGPT2's α_rel instead of matching an absolute norm. If ZymCTRL still
collapses the way `26` reported, that is now a real, defensible generalization result. If it
looks meaningfully different once the comparison is actually fair, that is equally real and
should replace the current §1h framing.

## What "matched" means here, precisely

1. Measure ProtGPT2's own residual-stream norm ‖h‖ at layer 12, using a set of real protein
   fragments as probes -- the anchor point every other headline number in this project is already
   calibrated to (`REFERENCE_NORM = 583.998` **is** ProtGPT2's own natural vector norm from `03`).
   This gives `anchor_alpha_rel = REFERENCE_NORM / h_ProtGPT2`.
2. Build ZymCTRL's utility-matched contrastive vectors at layers 12 and 30, exactly as `26` does
   (Fix 1, unchanged).
3. Measure ZymCTRL's own ‖h‖ at the same two layers, using the *same* probe fragments (forward
   passes need no conditioning -- any model can be probed with any input string).
4. Scale each ZymCTRL vector to `anchor_alpha_rel * h_ZymCTRL(layer)` instead of to 583.998. This
   is the same push, expressed relative to each model's own scale, rather than the same push in
   absolute units.

One honest caveat carried over from `locked-results.md`'s cross-check: ‖h‖ has shown ~16%
variation across different probe sets in this project so far. This notebook measures both models'
‖h‖ with the *same* probe set in the *same* run specifically to avoid adding that variance to the
comparison -- the anchor is self-consistent within this notebook, not imported from a different
one's measurement.

## Why ZymCTRL, and not one of the other three

Same architecture family as ProtGPT2 (GPT-2/CTRL, 36 layers, 1280-dim) -- the cleanest possible
comparison, with the fewest confounds beyond the one being fixed (no hook-path change, no
layer-count mismatch, no tokenization difference of the kind p-IgGen/Mistral-Prot have). It is
also the model whose original numbers are most salvageable (§1h's own text already notes ZymCTRL's
1x showed *some* effect, unlike ProtGPT2's clean "nothing at 1x" — worth knowing whether that
survives a fair comparison or was itself the confound).

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~100-130 minutes
(ProtGPT2 load + probe measurement, then `26`'s full recipe on ZymCTRL, plus folding).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding

torch.manual_seed(2026)
np.random.seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"
REFERENCE_NORM = 583.998   # ProtGPT2's own natural v_L norm (03) -- the anchor push in ABSOLUTE
                           # units. This notebook re-expresses it as a RELATIVE push instead.

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())


Setup complete. CUDA available: True


In [2]:
# --- Common probe set: real UniProt fragments, same accessions used throughout this project.
#     Used to measure BOTH models' residual-stream norm in this same run, so the anchor is
#     self-consistent rather than imported from a different notebook's measurement. ---

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

# Offline fallback -- real sequences already hardcoded in 03-ai4dd-uccs-baseline-test.ipynb
# (its positive_seqs). Used only if the UniProt fetch fails.
FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print()
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!! With Internet off the HuggingFace model downloads below will also fail.")
    print("!! Falling back to real fragments hardcoded in 03-ai4dd-uccs-baseline-test.ipynb.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_probe_set(reference_seqs, n_probes=40, frag_len=50, seed=7):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    probes = []
    for i in range(n_probes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        L = min(frag_len, len(seq))
        start = rng.randint(0, max(1, len(seq) - L + 1))
        probes.append(seq[start:start + L])
    return probes

probe_seqs = build_probe_set(reference_seqs, n_probes=40)
print(f"\nBuilt {len(probe_seqs)} common probe fragments from {len(reference_seqs)} source proteins.")

def measure_resid_norm(model, tokenizer, seqs, layer, hook_path="transformer.h"):
    obj = model
    for part in hook_path.split("."):
        obj = getattr(obj, part)
    layer_module = obj[layer]

    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = layer_module.register_forward_hook(hook)
    vals = []
    try:
        for seq in seqs:
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
            captured.clear()
            with torch.no_grad():
                model(**inputs)
            if "h" in captured:
                vals.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

print("Probe-measurement function ready.")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 40 common probe fragments from 12 source proteins.
Probe-measurement function ready.


In [3]:
# --- Step 1: measure the anchor. ProtGPT2's own layer-12 residual-stream norm, using the
#     probe set above. anchor_alpha_rel is what this project's "1x" has ACTUALLY meant on
#     ProtGPT2 all along -- everything else gets matched to this, not to the number 583.998. ---

print(f"Loading ProtGPT2 on {device} to measure the anchor...")
anchor_tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
anchor_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
anchor_model.eval()

h_protgpt2_l12 = measure_resid_norm(anchor_model, anchor_tokenizer, probe_seqs, 12,
                                    hook_path="transformer.h")
anchor_alpha_rel = REFERENCE_NORM / h_protgpt2_l12

print(f"ProtGPT2 layer 12 mean residual-stream norm (this run's probes): {h_protgpt2_l12:.2f}")
print(f"anchor_alpha_rel = {REFERENCE_NORM} / {h_protgpt2_l12:.2f} = {anchor_alpha_rel:.4f}")
print(f"  i.e. this project's '1x' has always meant displacing ProtGPT2 layer 12's hidden state")
print(f"  by ~{anchor_alpha_rel:.1%} of its own magnitude. ZymCTRL's vectors below are scaled to")
print(f"  produce the SAME {anchor_alpha_rel:.1%} displacement, not the same 583.998 absolute norm.")

del anchor_model, anchor_tokenizer
clear_gpu()
print("\nProtGPT2 freed from GPU.")


Loading ProtGPT2 on cuda to measure the anchor...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtGPT2 layer 12 mean residual-stream norm (this run's probes): 2919.92
anchor_alpha_rel = 583.998 / 2919.92 = 0.2000
  i.e. this project's '1x' has always meant displacing ProtGPT2 layer 12's hidden state
  by ~20.0% of its own magnitude. ZymCTRL's vectors below are scaled to
  produce the SAME 20.0% displacement, not the same 583.998 absolute norm.

ProtGPT2 freed from GPU.


In [4]:
# --- Same repetition/utility scoring as 24/26, unchanged. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0
        inputs = self.tokenizer([cleaned], return_tensors="pt", add_special_tokens=False).to(self.device)
        plddt, ptm = 0.0, 0.0
        try:
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
        except RuntimeError:
            clear_gpu()
        return plddt, ptm

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and ESMFold evaluator ready.")


Scoring functions and ESMFold evaluator ready.


In [5]:
# --- Identical to 26: EC-number candidate pool, ZymCTRL's own output-cleaning logic. ---

EC_LABELS = [
    "1.1.1.1", "1.1.1.2",
    "2.7.1.1", "2.7.1.2",
    "3.1.1.1", "3.5.1.4",
    "4.1.1.1", "4.2.1.1",
    "5.1.3.1", "5.3.1.9",
    "6.1.1.1", "6.3.2.1",
]

def build_ec_prompt_pool(labels, n_prompts, seed=11):
    rng = np.random.RandomState(seed)
    order = rng.permutation(n_prompts)
    return [labels[i % len(labels)] for i in order]

N_CANDIDATES = 200
candidate_prompts = build_ec_prompt_pool(EC_LABELS, N_CANDIDATES, seed=2601)
print(f"Built {len(candidate_prompts)} candidate EC-number prompts from {len(EC_LABELS)} real EC classes.")

print(f"Loading ZymCTRL on {device}...")
tokenizer = AutoTokenizer.from_pretrained("AI4PD/ZymCTRL")
plm_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
plm_model.eval()

EOS_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 1
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

def clean_zymctrl_output(decoded_text):
    seq_part = decoded_text.split("<sep>", 1)[1] if "<sep>" in decoded_text else decoded_text
    for tok in ["<start>", "<end>", "<|endoftext|>", "<pad>", " "]:
        seq_part = seq_part.replace(tok, "")
    return seq_part

def generate_natural(tokenizer, model, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    records = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, eos_token_id=EOS_ID, pad_token_id=PAD_ID
            )
        raw_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        seq = clean_zymctrl_output(raw_decoded)
        records.append({"prompt": prompt, "raw_decoded": raw_decoded, "sequence": seq, "gen_only": seq})
    clear_gpu()
    return records

print(f"=== Generating N={N_CANDIDATES} natural candidate sequences ===")
candidate_records = generate_natural(tokenizer, plm_model, candidate_prompts, max_len=50, seed=505)
print(f"Generated {len(candidate_records)} candidates.")

print("=== Freeing ZymCTRL while ESMFold folds the pool ===")
del plm_model
clear_gpu()

evaluator = StructuralEvaluatorPTM()
print("Folding and scoring the candidate pool...")
candidate_records = fold_records_ptm(candidate_records, evaluator)
del evaluator
clear_gpu()

valid_candidates = [r for r in candidate_records if r["plddt"] > 0.0]
print(f"\n{len(valid_candidates)}/{len(candidate_records)} candidates folded successfully.")
print(f"Mean pLDDT: {np.mean([r['plddt'] for r in valid_candidates]):.2f}, "
      f"natural collapse rate: {np.mean([r['collapse'] for r in valid_candidates]):.1%}")
print(f"(26 recorded: pool collapse 61.5%, mean pLDDT 57.28 -- close values here confirm the")
print(f" pool is comparable to that run.)")


Built 200 candidate EC-number prompts from 12 real EC classes.
Loading ZymCTRL on cuda...


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.88G [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding and scoring the candidate pool...

200/200 candidates folded successfully.
Mean pLDDT: 57.28, natural collapse rate: 61.5%
(26 recorded: pool collapse 61.5%, mean pLDDT 57.28 -- close values here confirm the
 pool is comparable to that run.)


In [6]:
# --- Same utility-matching as 24/26, unchanged. ---
QUANTILE = 0.30
UTILITY_TOLERANCE = 0.05

sorted_by_rep = sorted(valid_candidates, key=lambda r: r["repetition_score"])
n_side = max(10, int(len(sorted_by_rep) * QUANTILE))

d_minus_raw = sorted_by_rep[:n_side]
d_plus_raw = sorted_by_rep[-n_side:]

def utility_match(pool_a, pool_b, tolerance, max_iters=200):
    a, b = list(pool_a), list(pool_b)
    for _ in range(max_iters):
        mean_a = np.mean([r["utility_score"] for r in a])
        mean_b = np.mean([r["utility_score"] for r in b])
        gap = mean_a - mean_b
        if abs(gap) <= tolerance or min(len(a), len(b)) <= 15:
            break
        if gap > 0:
            a.sort(key=lambda r: -r["utility_score"])
            a.pop(0)
        else:
            b.sort(key=lambda r: r["utility_score"])
            b.pop(0)
    return a, b

d_plus, d_minus = utility_match(d_plus_raw, d_minus_raw, UTILITY_TOLERANCE)

mean_u_plus_after = np.mean([r["utility_score"] for r in d_plus])
mean_u_minus_after = np.mean([r["utility_score"] for r in d_minus])
mean_r_plus = np.mean([r["repetition_score"] for r in d_plus])
mean_r_minus = np.mean([r["repetition_score"] for r in d_minus])

print(f"After utility-matching:")
print(f"  D+ : n={len(d_plus):3d}  mean R(x)={mean_r_plus:.3f}  mean U(x)={mean_u_plus_after:.3f}")
print(f"  D- : n={len(d_minus):3d}  mean R(x)={mean_r_minus:.3f}  mean U(x)={mean_u_minus_after:.3f}")
print(f"  Remaining utility gap: {abs(mean_u_plus_after - mean_u_minus_after):.3f} "
      f"(target <= {UTILITY_TOLERANCE})")

if len(d_plus) < 15 or len(d_minus) < 15:
    print("\nWARNING: one side trimmed down very small.")


After utility-matching:
  D+ : n= 60  mean R(x)=0.968  mean U(x)=0.436
  D- : n= 60  mean R(x)=0.924  mean U(x)=0.451
  Remaining utility gap: 0.015 (target <= 0.05)


In [7]:
# --- Step 2+3: build the raw vectors AND measure ZymCTRL's own ‖h‖ at the same two layers,
#     using the SAME probe fragments as the ProtGPT2 anchor measurement. Then scale each vector
#     to anchor_alpha_rel * h_ZymCTRL(layer) instead of to REFERENCE_NORM. This is the one
#     substantive change from 26 -- everything else in this notebook is 26's recipe unchanged. ---

print(f"Reloading ZymCTRL on {device} to extract activations and measure ‖h‖...")
plm_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
plm_model.eval()

def get_mean_activation(model, tokenizer, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

d_plus_seqs = [r["sequence"] for r in d_plus]
d_minus_seqs = [r["sequence"] for r in d_minus]

steering_vectors = {}
h_zymctrl = {}
matched_norms = {}
for layer in [12, 30]:
    pos_acts = get_mean_activation(plm_model, tokenizer, d_plus_seqs, layer)
    neg_acts = get_mean_activation(plm_model, tokenizer, d_minus_seqs, layer)
    v_raw = pos_acts.mean(dim=0) - neg_acts.mean(dim=0)
    raw_norm = v_raw.norm().item()

    h = measure_resid_norm(plm_model, tokenizer, probe_seqs, layer, hook_path="transformer.h")
    h_zymctrl[layer] = h
    matched_norm = anchor_alpha_rel * h
    matched_norms[layer] = matched_norm

    v_matched = v_raw * (matched_norm / raw_norm)
    steering_vectors[layer] = v_matched.to(device)

    old_norm = REFERENCE_NORM  # what 26 used at this layer
    print(f"Layer {layer}: raw utility-matched v_L norm = {raw_norm:.4f}")
    print(f"  ZymCTRL ‖h‖ at this layer (this run's probes): {h:.2f}")
    print(f"  matched norm for anchor_alpha_rel={anchor_alpha_rel:.4f}: {matched_norm:.4f}")
    print(f"  (26 used a fixed {old_norm:.3f} here regardless of layer -- "
          f"{matched_norm / old_norm:.4f}x that value)")

print(f"\nSummary: ProtGPT2 anchor alpha_rel = {anchor_alpha_rel:.4f}")
print(f"ZymCTRL layer 12 matched norm: {matched_norms[12]:.2f}  (26 used {REFERENCE_NORM:.2f} -- "
      f"{matched_norms[12] / REFERENCE_NORM:.4f}x)")
print(f"ZymCTRL layer 30 matched norm: {matched_norms[30]:.2f}  (26 used {REFERENCE_NORM:.2f} -- "
      f"{matched_norms[30] / REFERENCE_NORM:.4f}x)")


Reloading ZymCTRL on cuda to extract activations and measure ‖h‖...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: AI4PD/ZymCTRL
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layer 12: raw utility-matched v_L norm = 3.3572
  ZymCTRL ‖h‖ at this layer (this run's probes): 51.26
  matched norm for anchor_alpha_rel=0.2000: 10.2531
  (26 used a fixed 583.998 here regardless of layer -- 0.0176x that value)
Layer 30: raw utility-matched v_L norm = 5.9438
  ZymCTRL ‖h‖ at this layer (this run's probes): 82.56
  matched norm for anchor_alpha_rel=0.2000: 16.5119
  (26 used a fixed 583.998 here regardless of layer -- 0.0283x that value)

Summary: ProtGPT2 anchor alpha_rel = 0.2000
ZymCTRL layer 12 matched norm: 10.25  (26 used 584.00 -- 0.0176x)
ZymCTRL layer 30 matched norm: 16.51  (26 used 584.00 -- 0.0283x)


In [8]:
# --- Steering conditions. Same hook mechanism as 26 -- model.transformer.h[L], no adaptation
#     needed (GPT-2 architecture family). Only the vector SCALE differs from 26. ---

def generate_with_vector_steering(model, tokenizer, target_layer, steering_vector, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    model.eval()
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        return (out[0] + v,)

    records = []
    for prompt in prompts:
        handle = model.transformer.h[target_layer].register_forward_hook(hook)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, eos_token_id=EOS_ID, pad_token_id=PAD_ID
            )
        handle.remove()
        raw_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        seq = clean_zymctrl_output(raw_decoded)
        records.append({"prompt": prompt, "sequence": seq, "entropy": calculate_entropy(seq)})
    clear_gpu()
    return records

steer_prompts = build_ec_prompt_pool(EC_LABELS, 250, seed=2602)

N_PER_CONDITION = 50
conditions = {}

print("=== CONTROL (unsteered) ===")
conditions["CONTROL"] = generate_with_vector_steering(
    plm_model, tokenizer, 12, None, steer_prompts[0:N_PER_CONDITION], seed=111)

print("=== L12_MATCHED_1x (alpha_rel-matched to ProtGPT2's anchor) ===")
conditions["L12_MATCHED_1x"] = generate_with_vector_steering(
    plm_model, tokenizer, 12, steering_vectors[12] * 1.0,
    steer_prompts[N_PER_CONDITION:2 * N_PER_CONDITION], seed=222)

print("=== L12_MATCHED_2x ===")
conditions["L12_MATCHED_2x"] = generate_with_vector_steering(
    plm_model, tokenizer, 12, steering_vectors[12] * 2.0,
    steer_prompts[2 * N_PER_CONDITION:3 * N_PER_CONDITION], seed=333)

print("=== L30_MATCHED_1x ===")
conditions["L30_MATCHED_1x"] = generate_with_vector_steering(
    plm_model, tokenizer, 30, steering_vectors[30] * 1.0,
    steer_prompts[3 * N_PER_CONDITION:4 * N_PER_CONDITION], seed=444)

print("=== L30_MATCHED_2x ===")
conditions["L30_MATCHED_2x"] = generate_with_vector_steering(
    plm_model, tokenizer, 30, steering_vectors[30] * 2.0,
    steer_prompts[4 * N_PER_CONDITION:5 * N_PER_CONDITION], seed=555)

for name, recs in conditions.items():
    print(f"{name}: {len(recs)} sequences generated")

print("=== Freeing ZymCTRL from GPU ===")
del plm_model
clear_gpu()


=== CONTROL (unsteered) ===
=== L12_MATCHED_1x (alpha_rel-matched to ProtGPT2's anchor) ===
=== L12_MATCHED_2x ===
=== L30_MATCHED_1x ===
=== L30_MATCHED_2x ===
CONTROL: 50 sequences generated
L12_MATCHED_1x: 50 sequences generated
L12_MATCHED_2x: 50 sequences generated
L30_MATCHED_1x: 50 sequences generated
L30_MATCHED_2x: 50 sequences generated
=== Freeing ZymCTRL from GPU ===


In [9]:
# --- Fold. Same VALID_AA filter, same pLDDT<60 threshold, PLUS a fold-success check on the
#     final conditions -- the exact gap 35-ai4dd-random-direction-control.ipynb's ORTHOGONAL_2x
#     artifact exposed (a candidate pool always checks "N/N folded successfully"; the final
#     steering conditions never did, anywhere in this project, until now). ---

evaluator2 = StructuralEvaluatorPTM()
for name in conditions:
    print(f"Folding {name}...")
    for r in conditions[name]:
        plddt, ptm = evaluator2.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["collapse"] = int(0.0 < plddt < 60.0)
del evaluator2
clear_gpu()

from scipy.stats import fisher_exact

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

print(f"\n{'Condition':18s} {'N':>4s} {'FoldOK':>7s} {'Entropy':>9s} {'pLDDT':>8s} {'pTM':>7s} {'Collapse%':>10s}")
print("-" * 74)
summary = {}
for name, recs in conditions.items():
    n_ok = sum(1 for r in recs if r["plddt"] > 0.0)
    ents = [r["entropy"] for r in recs]
    plddts = [r["plddt"] for r in recs if r["plddt"] > 0.0]
    ptms = [r["ptm"] for r in recs if r["plddt"] > 0.0]
    k = int(np.sum([r["collapse"] for r in recs]))
    n = len(recs)
    summary[name] = {"k": k, "n": n, "rate": k / n, "ci": wilson_ci(k, n), "fold_ok": n_ok}
    flag = "" if n_ok == n else "  <== check"
    print(f"{name:18s} {n:4d} {n_ok:3d}/{n:<3d} {np.mean(ents):9.3f} {np.mean(plddts):8.2f} "
          f"{np.mean(ptms):7.3f} {k / n * 100:9.1f}%{flag}")

if any(s["fold_ok"] < s["n"] for s in summary.values()):
    print()
    print("At least one condition had folds that never completed (plddt==0.0 returned). Per the")
    print("gap documented in locked-results.md SS1l, this can silently inflate the apparent")
    print("'not collapsed' count. Check the FoldOK column above before trusting any row where it")
    print("is well below N -- especially if that condition also generated unusually long output.")
else:
    print()
    print("All conditions folded successfully (FoldOK == N everywhere) -- no artifact of the kind")
    print("found in 35's ORTHOGONAL_2x condition.")


Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding L12_MATCHED_1x...
Folding L12_MATCHED_2x...
Folding L30_MATCHED_1x...
Folding L30_MATCHED_2x...

Condition             N  FoldOK   Entropy    pLDDT     pTM  Collapse%
--------------------------------------------------------------------------
CONTROL              50  50/50      3.770    57.22   0.326      66.0%
L12_MATCHED_1x       50  50/50      3.756    55.67   0.305      74.0%
L12_MATCHED_2x       50  50/50      3.763    54.46   0.282      72.0%
L30_MATCHED_1x       50  50/50      3.823    59.42   0.356      54.0%
L30_MATCHED_2x       50  50/50      3.822    56.86   0.325      60.0%

All conditions folded successfully (FoldOK == N everywhere) -- no artifact of the kind
found in 35's ORTHOGONAL_2x condition.


In [10]:
# --- The actual comparison this notebook exists to make. ---

print("=" * 100)
print("THE REPAIRED COMPARISON: ZymCTRL at matched relative push vs. ProtGPT2's anchor")
print("=" * 100)
print(f"ProtGPT2 anchor alpha_rel (this run's measurement): {anchor_alpha_rel:.4f}")
print()
print(f"{'Condition':18s} {'collapsed':>11s} {'rate':>8s} {'95% CI':>20s} {'alpha_rel':>10s}")
print("-" * 78)
for name, s in summary.items():
    lo, hi = s["ci"]
    ci_str = "[{:.1%}, {:.1%}]".format(lo, hi)
    if name == "CONTROL":
        a_rel = 0.0
    else:
        layer = 12 if "L12" in name else 30
        mult = 2.0 if name.endswith("2x") else 1.0
        a_rel = mult * matched_norms[layer] / h_zymctrl[layer]
    print(f"{name:18s} {s['k']:5d}/{s['n']:<5d} {s['rate']:7.1%} {ci_str:>20s} {a_rel:10.4f}")

print()
print("Read against 26's ORIGINAL (absolute-norm, mismatched) result at the same layers")
print("(locked-results.md §1h):")
print("  CONTROL 66.0% -> L12_FAIR_1x 86.0% -> L12_FAIR_2x 90.0% -> L30_FAIR_1x 72.0% -> L30_FAIR_2x 76.0%")
print(f"  (that 1x was actually alpha_rel ≈ 11.4 at layer 12 -- ~67x harder than this run's matched 1x)")
print()
print("This run (matched to ProtGPT2's own alpha_rel):")
ctrl = summary["CONTROL"]
print(f"  CONTROL {ctrl['rate']:.1%} -> "
      f"L12_MATCHED_1x {summary['L12_MATCHED_1x']['rate']:.1%} -> "
      f"L12_MATCHED_2x {summary['L12_MATCHED_2x']['rate']:.1%} -> "
      f"L30_MATCHED_1x {summary['L30_MATCHED_1x']['rate']:.1%} -> "
      f"L30_MATCHED_2x {summary['L30_MATCHED_2x']['rate']:.1%}")

print()
print("Read against ProtGPT2's OWN result at the same nominal alpha_rel (locked §1g, layer 12):")
print("  CONTROL 62.0% -> L12_FAIR_1x 60.0% -> L12_FAIR_2x 90.0%")

print()
print("=" * 100)
print("VERDICT")
print("=" * 100)

_, p_l12_1x = fisher_exact([[summary["L12_MATCHED_1x"]["k"], summary["L12_MATCHED_1x"]["n"] - summary["L12_MATCHED_1x"]["k"]],
                            [ctrl["k"], ctrl["n"] - ctrl["k"]]])
_, p_l12_2x = fisher_exact([[summary["L12_MATCHED_2x"]["k"], summary["L12_MATCHED_2x"]["n"] - summary["L12_MATCHED_2x"]["k"]],
                            [ctrl["k"], ctrl["n"] - ctrl["k"]]])

print(f"L12_MATCHED_1x vs CONTROL: p = {p_l12_1x:.4f}")
print(f"L12_MATCHED_2x vs CONTROL: p = {p_l12_2x:.4f}")
print()

no_effect_1x = p_l12_1x >= 0.05
big_effect_2x = p_l12_2x < 0.05 and summary["L12_MATCHED_2x"]["rate"] > ctrl["rate"] + 0.15

if no_effect_1x and big_effect_2x:
    print("  ==> ZymCTRL REPLICATES ProtGPT2's pattern once the push is actually matched:")
    print("      no detectable effect at the matched '1x', real collapse at matched '2x'.")
    print("      This is now a genuine, defensible 2-model generalization result -- the")
    print("      original §1h claim was directionally right but for the wrong reason (it was")
    print("      comparing a ~67x-too-hard push to ProtGPT2's moderate one). Update")
    print("      locked-results.md §1h and context-and-decisions.md §10.6 with this outcome.")
elif no_effect_1x and not big_effect_2x:
    print("  ==> ZymCTRL shows NO significant effect at either matched dose in this run.")
    print("      Either ZymCTRL is genuinely more robust to this magnitude of perturbation than")
    print("      ProtGPT2 (a real, reportable difference -- EC-conditioning may change how the")
    print("      residual stream encodes structural information), or 2x needs to go higher to")
    print("      find ZymCTRL's own threshold. Do not force this into 'replicates ProtGPT2' --")
    print("      report the difference honestly and consider a 3x/4x follow-up if time allows.")
else:
    print("  ==> ZymCTRL shows an effect even at the matched '1x'. This would mean ZymCTRL is")
    print("      MORE sensitive than ProtGPT2 to the same relative perturbation, at this layer.")
    print("      A real, reportable difference either way -- not a failed replication. Consider")
    print("      whether EC-conditioning narrows the natural output distribution enough that a")
    print("      smaller relative push is enough to leave it.")


THE REPAIRED COMPARISON: ZymCTRL at matched relative push vs. ProtGPT2's anchor
ProtGPT2 anchor alpha_rel (this run's measurement): 0.2000

Condition            collapsed     rate               95% CI  alpha_rel
------------------------------------------------------------------------------
CONTROL               33/50      66.0%       [52.2%, 77.6%]     0.0000
L12_MATCHED_1x        37/50      74.0%       [60.4%, 84.1%]     0.2000
L12_MATCHED_2x        36/50      72.0%       [58.3%, 82.5%]     0.4000
L30_MATCHED_1x        27/50      54.0%       [40.4%, 67.0%]     0.2000
L30_MATCHED_2x        30/50      60.0%       [46.2%, 72.4%]     0.4000

Read against 26's ORIGINAL (absolute-norm, mismatched) result at the same layers
(locked-results.md §1h):
  CONTROL 66.0% -> L12_FAIR_1x 86.0% -> L12_FAIR_2x 90.0% -> L30_FAIR_1x 72.0% -> L30_FAIR_2x 76.0%
  (that 1x was actually alpha_rel ≈ 11.4 at layer 12 -- ~67x harder than this run's matched 1x)

This run (matched to ProtGPT2's own alpha_rel):
  

In [11]:





# --- Persist. Same discipline as 35/36/37. ---
rows = []
for name, recs in conditions.items():
    layer = 12 if "L12" in name else (30 if "L30" in name else None)
    mult = 2.0 if name.endswith("2x") else (1.0 if name.endswith("1x") else 0.0)
    a_rel = 0.0 if layer is None else mult * matched_norms[layer] / h_zymctrl[layer]
    for i, r in enumerate(recs):
        rows.append({
            "condition": name, "idx": i, "layer": layer, "multiplier": mult, "alpha_rel": a_rel,
            "prompt": r["prompt"], "sequence": r["sequence"], "gen_length": len(r["sequence"]),
            "usable_length": sum(1 for a in r["sequence"] if a in VALID_AA),
            "entropy": r["entropy"], "plddt": r["plddt"], "ptm": r["ptm"], "collapse": r["collapse"],
        })
pd.DataFrame(rows).to_csv("zymctrl_matched_alpha_rel_sequences.csv", index=False)

pd.DataFrame([{
    "condition": n, "collapsed": s["k"], "n": s["n"], "collapse_rate": s["rate"],
    "ci_lo": s["ci"][0], "ci_hi": s["ci"][1], "fold_ok": s["fold_ok"],
} for n, s in summary.items()]).to_csv("zymctrl_matched_alpha_rel_summary.csv", index=False)

pd.DataFrame([{
    "anchor_alpha_rel": anchor_alpha_rel, "h_protgpt2_l12": h_protgpt2_l12,
    "h_zymctrl_l12": h_zymctrl[12], "h_zymctrl_l30": h_zymctrl[30],
    "matched_norm_l12": matched_norms[12], "matched_norm_l30": matched_norms[30],
    "reference_norm_used_by_26": REFERENCE_NORM,
}]).to_csv("zymctrl_matched_alpha_rel_calibration.csv", index=False)

print("Saved:")
print("  zymctrl_matched_alpha_rel_sequences.csv    (all sequences, all conditions)")
print("  zymctrl_matched_alpha_rel_summary.csv      (per-condition collapse rates + CIs + fold-success counts)")
print("  zymctrl_matched_alpha_rel_calibration.csv  (the anchor and matched norms, for the record)")
print()
print("Update notes/locked-results.md §1h with this run's verdict, and")
print("notes/context-and-decisions.md §10.6's 'not yet done' item once this is in.")


Saved:
  zymctrl_matched_alpha_rel_sequences.csv    (all sequences, all conditions)
  zymctrl_matched_alpha_rel_summary.csv      (per-condition collapse rates + CIs + fold-success counts)
  zymctrl_matched_alpha_rel_calibration.csv  (the anchor and matched norms, for the record)

Update notes/locked-results.md §1h with this run's verdict, and
notes/context-and-decisions.md §10.6's 'not yet done' item once this is in.
